In [ ]:
# !pip install jieba
# !pip install evaluate

In [ ]:
import jieba
import evaluate


In [ ]:
# English Example
predictions = ["hello, I don't understand."]
references = [
    ["hello, I don't know."]
]
bleu = evaluate.load("bleu")
results = bleu.compute(predictions=predictions, references=references)
print(results)

In [ ]:
# Chinese Example
space_sent = "因 此 ， 我 们 知 道 ， 洪 水 是 气 候 和 气 候 变 化 的 结 果 ， 不 同 的 发 生 在 不 同 的 情 况 下 发 生 了 不 同 的 。"
sent = ''
for word in space_sent:
    if word != " ":
        sent += word
print(sent)
words = list(jieba.cut(sent, cut_all=False)) # tokeize sentence using jieba
sent_pred = ''
for word in words:
    if sent_pred == '':
        sent_pred += word
    else:
        sent_pred += ' ' + word
print(sent_pred)

print()

sent = "洪水的产生是气候和河道共同作用的结果，不同河道形态下洪水产生的特点是不同的。"
print(sent)
words = list(jieba.cut(sent, cut_all=False)) # tokeize sentence using jieba
sent_ref = ''
for word in words:
    if sent_ref == '':
        sent_ref += word
    else:
        sent_ref += ' ' + word
print(sent_ref)

print()

predictions = [sent_pred]
references = [
    [sent_ref]
]
bleu = evaluate.load("bleu")
results = bleu.compute(predictions=predictions, references=references)
print(results)

In [ ]:
## reference code
# word segmentation on .txt file
# each line of this file includes one sentence
import jieba
import sys

input_file = sys.argv[1] # input text file
output_file = sys.argv[2] # output file to save results

with open(input_file,'r') as f2:
    sents = f2.readlines() # read lines of input file

lengths = []
with open(output_file,'w') as f1: # to save outputs
    for sent in sents: # one sentence at a time
        # f1.write(sent.strip()+'\t-->\t') # dump original sentence
        words = list(jieba.cut(sent, cut_all=False)) # tokeize sentence using jieba
        lengths.append(len(words)) # keep record of sentence lengths
        for word in words:
            if word == '\n':
                # f1.write(word.encode('utf-8'))
                f1.write(word)
            else:
            	# f1.write(word.encode('utf-8') + ' ') # dump tokenized sentence with tab separation
                f1.write(word + ' ')  
            # pdb.set_trace()


print('Minimum length: ',min(lengths))
print('Maximum length: ',max(lengths))
print('Average length: ',sum(lengths)/len(lengths))

In [ ]:
# Final BLEU evaluation on your own files 
import os
import jieba
import evaluate

# 1) Set file paths
# Prediction file: one predicted Chinese sentence per line.
# Reference file can be either:
#   - plain txt: one Chinese reference sentence per line
#   - tsv file: each line like "english\tchinese", and we will read the Chinese part
pred_file_path = "../runtime_core/save/predictions_test.txt"   # TODO: replace with your model output file
ref_file_path = "../runtime_core/data/en-cn/test.txt"          # test set provided by this project

# If your reference file is TSV (en\tzh), set this to True.
# If your reference file is plain Chinese txt (one sentence per line), set this to False.
ref_is_tsv = True


def zh_tokenize_to_spaced(sentence: str) -> str:
    """Tokenize Chinese sentence with jieba and join tokens by spaces for BLEU."""
    sentence = sentence.strip()
    if not sentence:
        return ""
    tokens = list(jieba.cut(sentence, cut_all=False))
    return " ".join(tokens)


def read_predictions(path: str):
    """Read predicted Chinese sentences (one line = one sentence)."""
    preds = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            preds.append(zh_tokenize_to_spaced(line))
    return preds


def read_references(path: str, is_tsv: bool):
    """Read reference Chinese sentences.

    Returns format required by evaluate BLEU:
    references = [[ref1], [ref2], ...]
    """
    refs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if is_tsv:
                parts = line.split("\t")
                if len(parts) < 2:
                    continue
                zh_sent = parts[1]
            else:
                zh_sent = line

            refs.append([zh_tokenize_to_spaced(zh_sent)])
    return refs


# 2) Sanity checks
if not os.path.exists(pred_file_path):
    raise FileNotFoundError(f"Prediction file not found: {pred_file_path}")
if not os.path.exists(ref_file_path):
    raise FileNotFoundError(f"Reference file not found: {ref_file_path}")

# 3) Batch preprocess
processed_preds = read_predictions(pred_file_path)
processed_refs = read_references(ref_file_path, ref_is_tsv)

if len(processed_preds) == 0:
    raise ValueError("Prediction file is empty after preprocessing.")
if len(processed_refs) == 0:
    raise ValueError("Reference file is empty after preprocessing.")

# Align lengths if they are not exactly the same.
n = min(len(processed_preds), len(processed_refs))
if len(processed_preds) != len(processed_refs):
    print(f"Warning: predictions({len(processed_preds)}) != references({len(processed_refs)}). Use first {n} pairs.")
processed_preds = processed_preds[:n]
processed_refs = processed_refs[:n]

# 4) Compute BLEU
bleu = evaluate.load("bleu")
results = bleu.compute(predictions=processed_preds, references=processed_refs)

print("最终测试集 BLEU 分数:", results)
print(f"有效评估句子数: {n}")

# Optional: print a few examples for report writing
show_k = min(5, n)
print("\n样例(分词后)：")
for i in range(show_k):
    print(f"[{i}] PRED: {processed_preds[i]}")
    print(f"[{i}] REF : {processed_refs[i][0]}")
    print("-")